In [ ]:
from pathlib import Path

import numpy  as np
import pandas as pd
from limosat_analysis import open_limosat
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
from cartopy.feature import LAND

DATA_PATH = Path("../data/201905_202007.parquet")  # Or a QC SQLite file
TIME_START = "2020-01-01"
TIME_STOP = "2020-01-04"  # Exclusive UTC boundary


## Read point observations

Install this repository in the notebook environment with `pip install -e ".[notebooks]"` from the repository root. Change `DATA_PATH` and the UTC dates in the first cell.

QC SQLite returns native observations. Displacement Parquet reconstructs linked endpoints, so singleton trajectories are unavailable. `time_utc` is timezone-aware UTC; `x_m` and `y_m` are EPSG:3413 metres. `trajectory_id` is the active trajectory ID within this source period; QC and original IDs are also retained when available.


In [ ]:
data = open_limosat(DATA_PATH, crs="EPSG:3413")
df = data.read_points(start=TIME_START, stop=TIME_STOP, linked_only=True)
df.head()


## Source information

The date interval includes `TIME_START` and excludes `TIME_STOP`. Point counts below describe the loaded interval. Parquet counts describe linked observations, and correlation remains missing in the older publisher format.


In [ ]:
df.attrs


## Plot all points for a given date range

In [ ]:
# Points were already filtered to the configured date range.
df1 = df
img_idx = df1.image_id.dropna().unique()
trj_idx = df1.trajectory_id.unique()
print(len(img_idx), "images in the date range")
print(len(trj_idx), "trajectories in the date range")

nps = ccrs.NorthPolarStereo(central_longitude=-45, true_scale_latitude=70.0)
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, projection=nps)
scat = ax.scatter(df1.x_m, df1.y_m, c=df1.image_id.to_numpy(dtype=float), cmap="jet", s=1)
plt.colorbar(scat, ax=ax, shrink=0.5)
ax.add_feature(LAND, zorder=10, edgecolor="black", facecolor="lightgray")
plt.show()


In [ ]:
# Calculate the number of points per image and per trajectory
points_per_image = df1.groupby('image_id').size()
points_per_trajectory = df1.groupby('trajectory_id').size()
fig, axs = plt.subplots(1,2, figsize=(12,3))
axs[0].hist(points_per_image, bins=50)
axs[1].hist(points_per_trajectory, bins=50)
plt.show()

## Plot the longest trajectories

In [ ]:
# Find IDs of trajectories with at least 10 points and plot the first 10 of them
long_trajectory_ids = points_per_trajectory[points_per_trajectory >=10].index.values
for trajectory_id, grp in list(df1[df1.trajectory_id.isin(long_trajectory_ids)].groupby('trajectory_id'))[:10]:
    plt.plot(grp.x_m, grp.y_m, '-')

## Plot consecutive ice-drift vectors

Read accepted consecutive links directly from Parquet, or derive them within each materialized trajectory in SQLite. This preserves QC breaks. Vector components are EPSG:3413 metres per second. Both endpoints must lie inside the plotting box.


In [ ]:
xlim = [0, 1e6]
ylim = [-1e6, 0]

vectors = data.read_displacements(start=TIME_START, stop=TIME_STOP)
in_box = (
    vectors.x0_m.between(*xlim) & vectors.y0_m.between(*ylim)
    & vectors.x1_m.between(*xlim) & vectors.y1_m.between(*ylim)
)
vectors = vectors.loc[in_box & (vectors.elapsed_seconds > 0)].copy()
u = vectors.dx_m / vectors.elapsed_seconds
v = vectors.dy_m / vectors.elapsed_seconds

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, projection=nps)
ax.quiver(vectors.x0_m, vectors.y0_m, u, v, angles="xy", scale_units="xy", scale=0.00001)
ax.add_feature(LAND, zorder=10, edgecolor="black", facecolor="lightgray")
plt.show()
